In [21]:
from value import Value
from typing import List
import random
from draw import draw_dot


class Neuron:

    def __init__(self, nin: int):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x: List[float]):
        # w * x + b
        # print(list(zip(self.w, x))) # pairing the input element with value and output as a tunple list
        act = sum([wi * xi for (wi, xi) in zip(self.w, x)], self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


def neuron_example():
    x = [2.0, 3.0]
    n = Neuron(2)
    # run the `__call__` function
    # zip(self.w,x)  ==> create a tuple for the element with same index then output to a list
    # like [(Value(data=0.31302155296131984, label=), 2.0), (Value(data=-0.4261160607177399, label=), 3.0)]
    print(n(x))


neuron_example()

Value(data=0.9887119180067312, label=)


![mlp](./images/layers.jpg)

In [ ]:
class Layer:
    def __init__(self, nin: int, nout: int):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x: int):
        outs = [n(x) for n in self.neurons]
        # return outs
        # output beautify
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        # return [p for neuron in self.neurons for p in neuron.parameters()] in short
        params: List[Value] = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps)
        return params


def layer_example():
    x = [2.0, 3.0]
    l = Layer(2, 3)
    print(l(x))


layer_example()

[Value(data=-0.9951322164163947, label=), Value(data=-0.6132902117912871, label=), Value(data=-0.18476893862127308, label=)]


In [ ]:
class MLP:
    # nin number of neurons in the input layer , nouts: each number stands for the number of neurons in each layer
    def __init__(self, nin: int, nouts: List[int]):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


def layerExample():
    x = [2.0, 3.0, -1.0]
    m = MLP(
        3, [4, 4, 1]
    )  # 3 neurons in input layers, 4 neurons first & second hidden layer, 1 neuron in the output layer
    out = m(x)
    print(out)
    return out


layerExample()

# draw_dot(layerExample())

Value(data=0.8568444175966475, label=)


Value(data=0.8568444175966475, label=)

In [ ]:
"""
scenario case:
with given MLP structure like 3,4,4,1
and given serveral inputs and desire outputs, how to tune the MLP parameters ?
input1: [2.0, 3.0, -1.0] wants 1.0
input2: [3.0, -1.0, 0.5] wants -1.0
input3: [0.5, 1.0, 1.0] wants -1.0
input4: [1.0, 1.0, -1.0] wants 1.0

"""


def make_inputs_and_targets():
    # input collections
    inputs: List[List[float]] = [
        [2.0, 3.0, -1.0],
        [3.0, -1.0, 0.5],
        [0.5, 1.0, 1.0],
        [1.0, 1.0, -1.0],
    ]
    targets: List[float] = [1.0, -1.0, -1.0, 1.0]
    return (inputs, targets)


def make_mlp():
    return MLP(3, [4, 4, 1])


def mlp_without_tuning():
    mlp = make_mlp()
    (inputs, _) = make_inputs_and_targets()

    youts = [mlp(x) for x in inputs]
    print(youts)
    return youts


mlp_without_tuning()


# each time mlp runs, it will output a value, which can be compare to the desires value,
# and the difference between those two values is called `loss`
# Our goal is to minimize the `loss`
def compute_loss():
    (inputs, targets) = make_inputs_and_targets()
    youts = mlp_without_tuning()
    # Loss Function, the MSE
    loss = [(yout - ytarget) ** 2 for (yout, ytarget) in zip(youts, targets)]
    return sum(loss)  # we want this value close to 0


compute_loss()

[Value(data=0.4997674025446651, label=), Value(data=0.9435638163204675, label=), Value(data=-0.15481972748039957, label=), Value(data=0.6538079805077015, label=)]
[Value(data=0.30114245482081603, label=), Value(data=-0.006755040944273376, label=), Value(data=0.21688057432928692, label=), Value(data=0.18707430018943047, label=)]


Value(data=3.616583942735967, label=)

In [ ]:
# we add the class method `paramters` for Neuron & Layer to collecti all the weight and bias
def make_mlp_gradient_descent():
    mlp = make_mlp()
    (inputs, targets) = make_inputs_and_targets()

    def forward_pass():
        youts = [mlp(x) for x in inputs]
        losses = [(yout - ytarget) ** 2 for (yout, ytarget) in zip(youts, targets)]
        loss: Value = sum(losses)
        return loss

    def gradient_descent(eta: float):
        ps = mlp.parameters()
        for p in ps:
            p.data += -1 * p.grad * eta

    return forward_pass, gradient_descent


def iterate():
    (recalculate_loss, gradient_descent) = make_mlp_gradient_descent()
    loss = recalculate_loss()
    print(loss)
    loss.backward()
    gradient_descent(0.01) # step can neither be too large or too low
    loss = recalculate_loss()
    print(loss)  # should be less


iterate()

Value(data=5.59912093957126, label=)
Value(data=5.086280664969494, label=)
